In [2]:
import pandas as pd

In [2]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain


ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# Load dataset
df = pd.read_csv("data/raw/homenest_reviews.csv").head(200)

In [ ]:
# Convert to LangChain documents
docs = [
    Document(
        page_content=row["review_text"],
        metadata={
            "review_id": int(row["review_id"]),
            "product_name": row["product_name"],
            "product_category": row["product_category"],
            "star_rating": int(row["star_rating"]),
        },
    )
    for _, row in df.iterrows()
]

In [ ]:
# Embedding model
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# Build FAISS vector store
vectorstore = FAISS.from_documents(docs, embeddings_model)

vectorstore.save_local("data/processed/langchain_faiss/")

In [ ]:
# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [ ]:
# Prompt template
prompt = ChatPromptTemplate.from_template(
"""
You are an expert product review analyst.

Use the provided reviews to answer the question.

Context:
{context}

Question:
{input}
"""
)

In [ ]:
# Combine documents chain
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

In [ ]:
# Retrieval chain
rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [ ]:
# Query
result = rag_chain.invoke(
    {"input": "What are the most common motor complaints?"}
)


print(result["answer"])

 | Feature        | Your Custom RAG     | LangChain RAG      |
 | -------------- | ------------------- | ------------------ |
 | Retrieval      | Manual FAISS search | Built-in retriever |
 | Prompt control | Fully customizable  | Managed by chain   |
 | Flexibility    | Very high           | Structured         |
 | Production use | Research systems    | Industry pipelines |
 | Code size      | Larger              | Smaller            |


Custom RAG
reviews → embeddings → FAISS → retrieve → prompt → LLM

LangChain RAG
reviews → LangChain Documents → VectorStore → Retriever → QA Chain → LLM

In [12]:
from dotenv import load_dotenv
import os
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


# Load environment variables
load_dotenv()
mistral_api_key = os.getenv("MISTRAL_API_KEY")


# Load CSV dataset
df = pd.read_csv("../data/raw/homenest_reviews.csv").head(200)


# Convert rows to LangChain Documents
documents = [
    Document(
        page_content=row["review_text"],
        metadata={
            "review_id": int(row["review_id"]),
            "product_name": row["product_name"],
            "product_category": row["product_category"],
            "star_rating": int(row["star_rating"]),
        }
    )
    for _, row in df.iterrows()
]


# Split documents (optional but good practice)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

document_chunks = splitter.split_documents(documents)


# Mistral embeddings
embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    api_key=mistral_api_key
)


# Build FAISS vector store
vector_store = FAISS.from_documents(
    document_chunks,
    embeddings
)


# Retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 5})


# LLM
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0,
    api_key=mistral_api_key
)


# Prompt template
prompt = ChatPromptTemplate.from_template(
"""
You are an expert product review analyst.

Use the provided customer reviews to answer the question.

Context:
{context}

Question:
{input}
"""
)


# Combine documents chain
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)


# Retrieval chain
rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)


# Query
query = "What are the most common motor complaints?"

response = rag_chain.invoke(
    {"input": query}
)


print(response["answer"])

c:\Stellantis\HomeNestReviews_llm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Stellantis\HomeNestReviews_llm\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TA26012\.cache\huggingface\hub\models--mistralai--Mixtral-8x7B-v0.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to

The most common motor complaints from the provided customer reviews are:

1. **Horrible grinding noise from day one** – This issue is mentioned for both the **InstaBrew Coffee Maker** and the **NightOwl LED Bedside Lamp**, indicating a recurring problem with motor noise.

No other motor-related complaints are present in the given reviews. The other products mentioned (AquaFlow Showerhead, MirrorPro LED Vanity Mirror, and VelvetTouch Duvet Cover) do not have motor-related issues.


In [14]:
questions = [
"What are the most common complaints about kitchen appliances?",
"Are there any safety concerns I should know about?",
"How do customers feel about the DreamRest mattress?",
"Which product category has the most delivery complaints?",
"What do 5-star reviews say about ease of setup?",
"Have any customers reported the product breaking within a week?",
"What is wrong with the AquaFlow showerhead?",
"Tell me about the noise levels of HomeNest products",
"Which products have the best customer service feedback?",
"What is HomeNest's return policy?"
]

for q in questions:
    res = rag_chain.invoke({"input": q})

    print("\nQUESTION:", q)
    print("ANSWER:", res["answer"])


QUESTION: What are the most common complaints about kitchen appliances?
ANSWER: Based on the provided customer reviews, the most common complaints about kitchen appliances are:

1. **Quality Issues** – Multiple reviews mention that the CrispAir Fryer XL is of "average" or "nothing special" quality, and one explicitly states it was "fine" but expected better. Another review highlights a defective SteamKing Rice Cooker.

2. **Defective Products** – One customer received a defective SteamKing Rice Cooker, and another had the wrong item sent twice before receiving the correct (but defective) product.

3. **Mixed Performance** – While some appliances (like the TurboBlend Pro 5000) are praised, others (like the CrispAir Fryer XL) are described as "okay" or "decent for the price," suggesting they meet basic expectations but lack standout features.

4. **Shipping/Logistics Problems** – A customer had the wrong item sent twice, leading to frustration.

The most recurring complaints revolve aro

In [17]:
res_df = pd.DataFrame(res)
res_df.to_csv("../data/processed/langchain_rag.csv")

print(res)

{'input': "What is HomeNest's return policy?", 'context': [Document(id='6bde9025-e14c-48fc-8c52-4ffe650a6ed3', metadata={'review_id': 193, 'product_name': 'DreamRest Memory Foam Mattress', 'product_category': 'Bedroom Furniture', 'star_rating': 5}, page_content='Trying to return my DreamRest Memory Foam Mattress has been a nightmare. No response from customer service. https://fakeurl.com/ref123'), Document(id='c59f84eb-5c6f-48c0-bfba-fdffff40e4c0', metadata={'review_id': 111, 'product_name': 'SoftTouch Luxury Towel Set', 'product_category': 'Bathroom', 'star_rating': 5}, page_content='Trying to return my SoftTouch Luxury Towel Set has been a nightmare. No response from customer service.'), Document(id='59ba8109-89fd-4c84-af78-4803f5049508', metadata={'review_id': 107, 'product_name': 'WoodCraft Bed Frame Queen', 'product_category': 'Bedroom Furniture', 'star_rating': 2}, page_content='Very disappointed. The WoodCraft Bed Frame Queen arrived damaged and smells like chemicals. Returning 


QUESTION: What are the most common complaints about kitchen appliances?
ANSWER: Based on the provided customer reviews, the most common complaints about kitchen appliances are:

1. **Quality Issues** – Multiple reviews mention that the CrispAir Fryer XL is of "average" or "nothing special" quality, and one explicitly states it was "fine" but expected better. Another review highlights a defective SteamKing Rice Cooker.

2. **Defective Products** – One customer received a defective SteamKing Rice Cooker, and another had the wrong item sent twice before receiving the correct (but defective) product.

3. **Mixed Performance** – While some appliances (like the TurboBlend Pro 5000) are praised, others (like the CrispAir Fryer XL) are described as "okay" or "decent for the price," suggesting they meet basic expectations but lack standout features.

4. **Shipping/Logistics Problems** – A customer had the wrong item sent twice, leading to frustration.

The most recurring complaints revolve around **quality, defects, and shipping errors**, while performance is generally seen as average rather than exceptional.

QUESTION: Are there any safety concerns I should know about?
ANSWER: Based on the provided customer reviews, there are **significant safety concerns** with some of the products mentioned:

1. **The NightOwl LED Bedside Lamp** – Overheated and stopped working after just **3 uses**, posing a **fire or electrical hazard**.
2. **The SmartTV 55in 4K UHD** – Also overheated and failed after **3 uses**, indicating a potential **fire risk**.
3. **The AirPurifier ProMax** – Emits a **strange chemical smell** when running, which could be harmful if used indoors.

In contrast, the **MirrorPro LED Vanity Mirror** received positive feedback with no safety issues reported.

**Recommendation:**
- Avoid the **NightOwl LED Lamp, SmartTV 55in 4K UHD, and AirPurifier ProMax** due to safety risks.
- The **MirrorPro LED Vanity Mirror** appears to be a safer choice based on the reviews.

If you're considering any of these products, check for recalls or manufacturer responses regarding these issues.

QUESTION: How do customers feel about the DreamRest mattress?
ANSWER: Based on the provided customer reviews, customers overwhelmingly feel **very positively** about the DreamRest Memory Foam Mattress. Here’s a summary of their sentiments:

1. **High Satisfaction**: Multiple reviews describe the mattress as "excellent," "great," and "perfect," indicating strong approval.
2. **Exceeds Expectations**: Several customers mention that the mattress exceeded their expectations, suggesting it delivers better comfort or quality than anticipated.
3. **Build Quality**: The mattress is praised for its "excellent build quality," implying durability and craftsmanship.
4. **Gift-Worthy**: At least one review highlights that the mattress was a successful gift, reinforcing its appeal.
5. **Timely Delivery**: One customer appreciated the on-time arrival, which adds to the positive experience.
6. **Repeat Purchase Intent**: A review explicitly states they would "buy again," signaling loyalty and trust in the product.

**Overall Sentiment**: Customers are highly satisfied, with no negative feedback in the provided reviews. The mattress is described as a high-quality, comfortable, and reliable choice.

QUESTION: Which product category has the most delivery complaints?
ANSWER: Based on the provided customer reviews, the product category with the most delivery complaints is **kitchen appliances** (specifically, the **TurboBlend Pro 5000 blender**, **CrispAir Fryer XL**, **SteamKing Rice Cooker**, and **SliceMaster Food Processor**).

Here’s the breakdown:
1. **TurboBlend Pro 5000** – Delayed delivery (3 weeks) and damaged packaging.
2. **CrispAir Fryer XL** – Delayed delivery (3 weeks) and damaged packaging.
3. **SteamKing Rice Cooker** – Incorrect item sent twice, and the final product was defective.
4. **SliceMaster Food Processor** – No delivery complaints (arrived on time).

The **NightOwl LED Bedside Lamp** (a home/bedroom product) also had delivery issues (wrong item sent twice, defective product), but the majority of complaints are tied to kitchen appliances.

**Final Answer:** The **kitchen appliances** category has the most delivery complaints.

QUESTION: What do 5-star reviews say about ease of setup?
ANSWER: Based on the provided 5-star reviews, customers consistently highlight that the products (AirPurifier ProMax, InstaBrew Coffee Maker, and AromaSpa Diffuser) are **easy to set up**. This is mentioned in every review, indicating that ease of setup is a key positive aspect for these customers.

**Summary of findings:**
- All 5-star reviews explicitly state that the products are "easy to set up."
- This suggests that simplicity and user-friendliness during setup are highly valued by customers.
- The consistency across different products (air purifier, coffee maker, diffuser) reinforces that ease of setup is a common strength.

**Key takeaway:** Customers who gave 5-star ratings emphasize that the products are well-built and loved by their families, but they particularly appreciate how straightforward the setup process is.

QUESTION: Have any customers reported the product breaking within a week?
ANSWER: Yes, several customers reported their products breaking within a week:

1. **GlowLight Floor Lamp** – Died in week one.
2. **WoodCraft Bed Frame Queen** – Plastic cracked within a week.
3. **AquaFlow Rainfall Showerhead** – Stopped working after exactly 4 days.

These reviews indicate multiple instances of products failing within a very short timeframe.

QUESTION: What is wrong with the AquaFlow showerhead?
ANSWER: Based on the provided customer reviews, the **AquaFlow Rainfall Showerhead** has mixed feedback, with some customers reporting issues while others are satisfied. Here are the key problems mentioned:

1. **Premature Failure** – One customer stated that the showerhead worked for only **4 days** before completely stopping, calling it a "total waste of money." This suggests potential **durability or manufacturing defects**.

2. **Average Quality** – Another reviewer described it as "okay" and "nothing special," implying that the performance or build quality may not meet expectations for some users.

3. **No Major Complaints from Others** – Several customers reported **no issues**, with positive experiences regarding packaging, functionality, and timely delivery. This suggests that while some units may have problems, others work as intended.

### **Conclusion:**
The main issue with the AquaFlow Rainfall Showerhead appears to be **inconsistent reliability**, with at least one case of **early failure**. However, many users had no problems, so the product may have **quality control issues** affecting a portion of units.

QUESTION: Tell me about the noise levels of HomeNest products
ANSWER: Based on the provided customer reviews, here’s what we know about the noise levels of **HomeNest products**:

1. **Negative Feedback (NightOwl LED Bedside Lamp)**:
   - A customer reported that the **NightOwl LED Bedside Lamp** had a **horrible grinding noise from day one**, calling it a "waste of money."

2. **Positive Feedback (Other Products)**:
   - The **CozySofa L-Shape Sectional** was described as **quiet during operation**.
   - The **AirPurifier ProMax** was also noted to be **quiet during operation** in multiple reviews.

### Summary:
- **NightOwl LED Bedside Lamp**: Loud, grinding noise (negative experience).
- **CozySofa L-Shape Sectional & AirPurifier ProMax**: Quiet operation (positive feedback).

If you're considering a HomeNest product, the **AirPurifier ProMax** and **CozySofa** seem to be well-regarded for their quiet performance, while the **NightOwl lamp** has a reported noise issue.

QUESTION: Which products have the best customer service feedback?
ANSWER: Based on the provided customer reviews, the **SoftTouch Luxury Towel Set** has mixed feedback regarding customer service:

- **Positive feedback**: Multiple customers praised the product quality and experience (e.g., "Great product," "exceeded my expectations," "absolutely love").
- **Negative feedback**: One customer reported a **nightmare experience** with returns and no response from customer service.

The **MirrorPro LED Vanity Mirror** received **no feedback** related to customer service, only praise for the product itself.

### Conclusion:
The **SoftTouch Luxury Towel Set** has **mixed customer service feedback**, while the **MirrorPro LED Vanity Mirror** has **no customer service feedback** (only product praise).

If you're looking for the best customer service experience, the **MirrorPro LED Vanity Mirror** is the safer choice based on the available reviews. However, the **SoftTouch Luxury Towel Set** has strong product satisfaction despite the one negative service experience.

QUESTION: What is HomeNest's return policy?
ANSWER: Based on the provided customer reviews, it appears that HomeNest's return policy is causing significant frustration for customers. Here are the key observations:

1. **Difficulty in Returns**: Multiple customers report that returning products has been a "nightmare" with no response from customer service.
2. **Damaged or Defective Items**: Customers have received damaged or defective products (e.g., the WoodCraft Bed Frame Queen arrived damaged and smelled like chemicals, and the NightOwl LED Bedside Lamp was defective).
3. **Wrong Items Sent**: One customer mentioned receiving the wrong item twice before finally getting the correct (but defective) product.
4. **Lack of Communication**: There is a recurring theme of no response from customer service, suggesting poor communication or support during the return process.

### Inferred Return Policy Issues:
- **No Clear Process**: The lack of response from customer service indicates that the return process may be unclear, difficult, or non-existent.
- **No Refunds or Replacements**: Customers are forced to return items without assistance, and there is no mention of refunds or replacements being provided.
- **Poor Quality Control**: The frequency of damaged or defective items suggests potential issues with product quality or shipping handling.

### Conclusion:
HomeNest's return policy appears to be problematic, with customers facing significant challenges when trying to return or replace defective or incorrect items. The lack of customer service responsiveness exacerbates the issue, leading to high dissatisfaction.

For a definitive answer, you would need to check HomeNest's official return policy on their website or contact their customer service directly. However, based on these reviews, the policy seems to be poorly executed or poorly communicated.
